# Day 1 — Real-Time Dataset RAG Agent with Chroma

This notebook restructures the original PDF-based program into a **dataset-based RAG agent**.

### Pipeline

`Dataset → LangChain Documents → Chunks → Gemini Embeddings → Chroma → Retriever → Tools → Gemini Agent`

It also keeps and improves the important ideas from the original notebook:

- Chroma persistence
- metadata inspection and metadata filtering
- calculator tool
- current-time tool
- agent memory/checkpointing
- streaming agent execution
- follow-up questions using the same thread
- clean error handling
- no hard-coded API key

> **Security:** never put a real Google API key directly into a notebook. Set `GOOGLE_API_KEY` in your environment or enter it interactively when prompted.


In [1]:
# ============================================================
# 0. INSTALL / UPDATE PACKAGES
# ============================================================

# Run this once in your Jupyter environment.
# If your environment already has the packages, you can skip it.

%pip install -qU langchain langchain-google-genai langchain-chroma langchain-text-splitters langgraph


Note: you may need to restart the kernel to use updated packages.


In [2]:
# ============================================================
# 1. IMPORTS + API KEY CHECK
# ============================================================

import ast
import datetime as dt
import getpass
import os
from pathlib import Path
from typing import Optional

from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings,
)
from langchain_chroma import Chroma
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent


# Never hard-code the API key in the notebook.
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass(
        "Enter your Google API key (input is hidden): "
    )

print("Environment ready.")


Enter your Google API key (input is hidden):  ········


Environment ready.


## 2. Dataset

The original notebook loaded a PDF. We replace that source with a small structured dataset made of LangChain `Document` objects.

Each document has:

- `page_content` → the text we want the RAG system to search
- `metadata` → labels we can later filter with Chroma

The dataset intentionally contains AI/RAG information **and** business/financial information so the calculator tool can demonstrate a complete retrieval → calculation workflow.


In [6]:
# ============================================================
# 2. DATASET → LANGCHAIN DOCUMENTS
# ============================================================

docs = [
    Document(
        page_content=(
            "Artificial intelligence allows computers to perform tasks "
            "that normally require human intelligence, including "
            "reasoning, language understanding, and pattern recognition."
        ),
        metadata={"source": "learning_dataset", "topic": "AI", "doc_id": "ai-001"},
    ),

    Document(
        page_content=(
            "Machine learning is a branch of artificial intelligence "
            "where computers learn patterns from data instead of being "
            "programmed with every rule explicitly."
        ),
        metadata={
            "source": "learning_dataset",
            "topic": "Machine Learning",
            "doc_id": "ml-001",
        },
    ),

    Document(
        page_content=(
            "Retrieval Augmented Generation, or RAG, allows a language "
            "model to retrieve relevant external information before "
            "generating an answer. This can help ground answers in "
            "a private knowledge base."
        ),
        metadata={"source": "learning_dataset", "topic": "RAG", "doc_id": "rag-001"},
    ),

    Document(
        page_content=(
            "Vector databases store vector representations of information "
            "and support similarity-based retrieval. Chroma can store "
            "documents, embeddings, IDs, and metadata for retrieval."
        ),
        metadata={
            "source": "learning_dataset",
            "topic": "Vector Database",
            "doc_id": "vec-001",
        },
    ),

    Document(
        page_content=(
            "Acme Learning reported total revenue of 2500000 dollars "
            "for the example fiscal year. This number is synthetic "
            "training data created for this notebook."
        ),
        metadata={
            "source": "learning_dataset",
            "topic": "Finance",
            "doc_id": "fin-001",
        },
    ),

    Document(
        page_content=(
            "Acme Learning had 125 employees in the example fiscal year. "
            "This number is synthetic training data created for this notebook."
        ),
        metadata={
            "source": "learning_dataset",
            "topic": "Finance",
            "doc_id": "fin-002",
        },
    ),

    Document(
        page_content=(
            "The example company increased its annual research budget "
            "to 400000 dollars. This number is synthetic training data "
            "created for this notebook."
        ),
        metadata={
            "source": "learning_dataset",
            "topic": "Finance",
            "doc_id": "fin-003",
        },
    ),

    Document(
        page_content=(
            "For this learning project, metadata contains source, topic, "
            "and doc_id. Metadata can be used to narrow Chroma searches "
            "before returning relevant documents."
        ),
        metadata={
            "source": "learning_dataset",
            "topic": "Metadata",
            "doc_id": "meta-001",
        },
    ),
]

print(f"Loaded {len(docs)} documents.")


Loaded 8 documents.


In [7]:
# ============================================================
# 3. CHUNKING
# ============================================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = splitter.split_documents(docs)

print(f"Documents: {len(docs)}")
print(f"Chunks:    {len(chunks)}")

print("\nExample chunk:")
print(chunks[0].page_content)

print("\nExample metadata:")
print(chunks[0].metadata)


Documents: 8
Chunks:    8

Example chunk:
Artificial intelligence allows computers to perform tasks that normally require human intelligence, including reasoning, language understanding, and pattern recognition.

Example metadata:
{'source': 'learning_dataset', 'topic': 'AI', 'doc_id': 'ai-001'}


## 4. Gemini Embeddings + Persistent Chroma

`gemini-embedding-2-preview` converts text into vectors.

Chroma stores those vectors together with the document text and metadata.

The database is persisted under `./chroma_db`, so the collection can survive a notebook restart.


In [8]:
# ============================================================
# 4. EMBEDDINGS + PERSISTENT CHROMA
# ============================================================

EMBEDDING_MODEL = "gemini-embedding-2-preview"
CHROMA_DIR = "./chroma_db"
COLLECTION_NAME = "day1_dataset_rag"

embeddings = GoogleGenerativeAIEmbeddings(
    model=EMBEDDING_MODEL
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
    collection_name=COLLECTION_NAME,
)

print("Chroma is ready.")
print(f"Persistent directory: {Path(CHROMA_DIR).resolve()}")


Chroma is ready.
Persistent directory: E:\AI-ML\chroma_db


In [9]:
# ============================================================
# 5. INSPECT CHROMA + METADATA
# ============================================================

stored = vectorstore.get()

print("Available Chroma fields:")
print(stored.keys())

print("\nStored IDs:")
print(stored["ids"])

print("\nStored metadata:")
for metadata in stored["metadatas"]:
    print(metadata)

print("\nStored document count:")
print(len(stored["documents"]))


Available Chroma fields:
dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

Stored IDs:
['8307184e-4e4c-4e4f-8ed4-c0dda5121a5f', 'e26e4a29-75d4-4492-b76c-b1efc2f12978', 'cb210a6c-74c7-46ee-a437-79befab3cd44', '23b8687b-7d51-4ca3-b7de-4ab2b66ef8c2', 'ff0f2b4c-5a9d-47b4-9418-a10565177cac', 'f403d83e-02b8-4e81-9cf9-9f749a352695', '735aa0c3-8a84-48ef-91cc-fb6ac8dd7cd3', '12226ee4-7b51-4d23-afa2-f392c0ccde03', 'c5c15fbe-f39a-4ecb-a58f-51056228b567', '786edc0a-c73a-435f-8672-1d4672bf07eb', 'a35c2e25-5142-4269-85ad-2b32047ba354', 'b4062512-fd00-4723-942a-11411eaaea5f', 'e7eb0439-8048-4d1f-a7ca-7d583afe902b', 'bd2d5f67-904a-4f3e-b785-fb7e2d318079', '94c8754b-a5e8-4abe-9516-f8216ef1c5cf', '0ee156b0-a6e7-4a62-8150-83b1eed338f9']

Stored metadata:
{'source': 'learning_dataset', 'doc_id': 'ai-001', 'topic': 'AI'}
{'topic': 'Machine Learning', 'source': 'learning_dataset', 'doc_id': 'ml-001'}
{'source': 'learning_dataset', 'topic': 'RAG', 'doc_id': 'rag-001'}
{'

## 6. Chroma Metadata Queries

These are useful for learning the exact metadata queries you asked about.

Examples:

- retrieve only `Finance`
- retrieve only `RAG`
- retrieve only one `doc_id`
- combine semantic similarity with metadata filtering


In [10]:
# ============================================================
# 6. METADATA QUERIES
# ============================================================

# A. Get documents whose metadata topic is Finance
finance_docs = vectorstore.get(
    where={"topic": "Finance"}
)

print("Finance metadata query:")
for text, metadata in zip(
    finance_docs["documents"],
    finance_docs["metadatas"]
):
    print(metadata, "->", text[:100])

print("\n" + "-" * 70)

# B. Get only the RAG document
rag_docs = vectorstore.get(
    where={"topic": "RAG"}
)

print("RAG metadata query:")
for text, metadata in zip(
    rag_docs["documents"],
    rag_docs["metadatas"]
):
    print(metadata, "->", text)

print("\n" + "-" * 70)

# C. Get one exact document by metadata
one_doc = vectorstore.get(
    where={"doc_id": "fin-001"}
)

print("Exact doc_id query:")
print(one_doc["metadatas"])
print(one_doc["documents"])


Finance metadata query:
{'doc_id': 'fin-001', 'source': 'learning_dataset', 'topic': 'Finance'} -> Acme Learning reported total revenue of 2500000 dollars for the example fiscal year. This number is 
{'topic': 'Finance', 'doc_id': 'fin-002', 'source': 'learning_dataset'} -> Acme Learning had 125 employees in the example fiscal year. This number is synthetic training data c
{'doc_id': 'fin-003', 'topic': 'Finance', 'source': 'learning_dataset'} -> The example company increased its annual research budget to 400000 dollars. This number is synthetic
{'doc_id': 'fin-001', 'source': 'learning_dataset', 'topic': 'Finance'} -> Acme Learning reported total revenue of 2500000 dollars for the example fiscal year. This number is 
{'topic': 'Finance', 'source': 'learning_dataset', 'doc_id': 'fin-002'} -> Acme Learning had 125 employees in the example fiscal year. This number is synthetic training data c
{'topic': 'Finance', 'source': 'learning_dataset', 'doc_id': 'fin-003'} -> The example company i

In [11]:
# ============================================================
# 7. RETRIEVER
# ============================================================

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever ready.")


Retriever ready.


## 8. Tools

The agent gets three tools:

1. `search_dataset` — semantic RAG search, with an optional metadata topic filter
2. `calculator` — safe arithmetic without Python `eval`
3. `get_current_time` — returns the machine's current local date/time

The search tool also returns metadata, so you can see where the answer came from.


In [16]:
# ============================================================
# 8A. DATASET SEARCH TOOL
# ============================================================

@tool
def search_dataset(
    query: str,
    topic: Optional[str] = None,
) -> str:
    """Search the dataset using semantic similarity.

    Optionally filter by metadata topic.
    Examples:
    - query='What is RAG?'
    - query='What was the revenue?', topic='Finance'
    """

    if topic:
        results = vectorstore.similarity_search(
            query,
            k=3,
            filter={"topic": topic},
        )
    else:
        results = retriever.invoke(query)

    if not results:
        return "No relevant information was found in the dataset."

    formatted = []

    for doc in results:
        formatted.append(
            f"[metadata={doc.metadata}]\n{doc.page_content}"
        )

    return "\n\n".join(formatted)


In [17]:
# ============================================================
# 8B. SAFE CALCULATOR TOOL
# ============================================================

def safe_eval(expression: str):
    """Evaluate a restricted arithmetic expression safely."""

    allowed_operators = {
        ast.Add: lambda a, b: a + b,
        ast.Sub: lambda a, b: a - b,
        ast.Mult: lambda a, b: a * b,
        ast.Div: lambda a, b: a / b,
        ast.USub: lambda a: -a,
        ast.Pow: lambda a, b: a ** b,
    }

    def _eval(node):
        if isinstance(node, ast.Constant) and isinstance(
            node.value, (int, float)
        ):
            return node.value

        if isinstance(node, ast.BinOp):
            operator_fn = allowed_operators.get(type(node.op))
            if operator_fn is None:
                raise TypeError("Unsupported math operation")
            return operator_fn(
                _eval(node.left),
                _eval(node.right),
            )

        if isinstance(node, ast.UnaryOp):
            operator_fn = allowed_operators.get(type(node.op))
            if operator_fn is None:
                raise TypeError("Unsupported unary operation")
            return operator_fn(_eval(node.operand))

        raise TypeError("Only numeric arithmetic is allowed")

    tree = ast.parse(expression, mode="eval").body
    return _eval(tree)


@tool
def calculator(expression: str) -> str:
    """Calculate a numeric arithmetic expression.

    Example: '2500000 * 1.15'
    """
    try:
        result = safe_eval(expression)
        return f"Calculation Result: {result}"
    except Exception as exc:
        return f"Calculator error: {exc}"


In [18]:
# ============================================================
# 8C. REAL-TIME CLOCK TOOL
# ============================================================

@tool
def get_current_time() -> str:
    """Return the current local date and time of this Python runtime."""

    now = dt.datetime.now().astimezone()

    return now.strftime(
        "%Y-%m-%d %H:%M:%S %Z (UTC%z)"
    )


## 9. Gemini Agent + Memory

The original notebook used the older `create_react_agent` API. This version uses the current LangChain `create_agent` style and keeps an in-memory checkpointer for conversation threads.

A `thread_id` is the conversation's memory key.


In [19]:
# ============================================================
# 9. CREATE GEMINI AGENT + MEMORY
# ============================================================

MODEL_NAME = "gemini-3.5-flash-lite"

model = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    temperature=0,
)

memory = InMemorySaver()

tools = [
    search_dataset,
    calculator,
    get_current_time,
]

SYSTEM_PROMPT = """
You are a helpful RAG assistant.

Rules:
1. Use search_dataset for factual questions about the dataset.
2. If the user asks about a number contained in the dataset,
   retrieve it before calculating with it.
3. Use calculator for arithmetic instead of doing arithmetic mentally.
4. Use get_current_time when the user asks for the current date/time.
5. Do not invent information that is absent from the dataset.
6. When useful, mention the metadata/source returned by retrieval.
7. Keep answers clear and beginner-friendly.
"""

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=memory,
)

config = {
    "configurable": {
        "thread_id": "day1-demo-session"
    }
}

print("Agent initialized.")


Agent initialized.


## 10. First Real-Time RAG Test

This question demonstrates the full chain:

`question → agent → search tool → Chroma → retrieved revenue → calculator → answer`


In [20]:
# ============================================================
# 10. REAL-TIME RAG + CALCULATOR TEST
# ============================================================

user_query = (
    "Find the example company's total revenue in the dataset. "
    "Then calculate what the revenue would be after a 15% increase. "
    "Show the original amount, calculation, and final amount."
)

print("USER:")
print(user_query)
print("\nLIVE AGENT STREAM:\n")

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": user_query}]},
    config=config,
    stream_mode="updates",
):
    for node_name, update in chunk.items():
        print(f"\n[{node_name.upper()}]")

        messages = update.get("messages", [])

        for message in messages:
            if getattr(message, "tool_calls", None):
                for call in message.tool_calls:
                    print(f"Tool requested: {call['name']}")
                    print(f"Arguments: {call['args']}")

            elif getattr(message, "type", "") == "tool":
                print("Tool result:")
                print(str(message.content)[:500])

            elif getattr(message, "content", None):
                print("Agent:")
                print(message.content)


USER:
Find the example company's total revenue in the dataset. Then calculate what the revenue would be after a 15% increase. Show the original amount, calculation, and final amount.

LIVE AGENT STREAM:



E:\AI-ML\.venv310\lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



[MODEL]
Tool requested: search_dataset
Arguments: {'query': 'revenue'}

[TOOLS]
Tool result:
[metadata={'source': 'learning_dataset', 'topic': 'Finance', 'doc_id': 'fin-001'}]
Acme Learning reported total revenue of 2500000 dollars for the example fiscal year. This number is synthetic training data created for this notebook.

[metadata={'doc_id': 'fin-001', 'topic': 'Finance', 'source': 'learning_dataset'}]
Acme Learning reported total revenue of 2500000 dollars for the example fiscal year. This number is synthetic training data created for this notebook.

[metadata={'source': 'learning


E:\AI-ML\.venv310\lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



[MODEL]
Tool requested: calculator
Arguments: {'expression': '2500000 * 1.15'}

[TOOLS]
Tool result:
Calculation Result: 2875000.0


E:\AI-ML\.venv310\lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



[MODEL]
Agent:
[{'type': 'text', 'text': "Based on the dataset (Source: `learning_dataset`, Topic: `Finance`), here is the breakdown of the example company's revenue and the 15% increase calculation:\n\n* **Original Amount:** $2,500,000 (Acme Learning's total revenue for the example fiscal year)\n* **Calculation:** $2,500,000 \\times 1.15$ \n* **Final Amount:** $2,875,000$", 'extras': {'signature': 'El4KXAERTTIPMkZkqPzYJydS5lC+nAQ8Unl202/ZYTuPYbjxl84P1OhFmCXF6zN6XEcfzpGj+Z/we0PbHhEYo3hnkdVgbXOeEKc1aJaFEif362ASNdOai3NEp5q/MEG9'}}]


In [21]:
# ============================================================
# 11. NORMAL QUERY + METADATA FILTER
# ============================================================

query = "Explain RAG in simple words."

result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": query}
        ]
    },
    config=config,
)

print(result["messages"][-1].content)

print("\nDirect metadata-filtered retrieval:")
print(
    search_dataset.invoke(
        {
            "query": "What is retrieval augmented generation?",
            "topic": "RAG",
        }
    )
)


E:\AI-ML\.venv310\lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': '**RAG** stands for **Retrieval-Augmented Generation**. \n\nThink of it like an open-book exam for an AI:\n\n1. **Retrieval (Finding the info):** When you ask a question, the system first searches a specific collection of documents, databases, or files to find the exact information needed to answer your question.\n2. **Generation (Writing the answer):** The AI takes the retrieved information and uses it to write a clear, accurate, and easy-to-understand response in its own words.\n\nInstead of relying solely on what the AI happened to memorize during its initial training (which can sometimes be outdated or incorrect), RAG checks reliable sources first so it can give you a factual and up-to-date answer!', 'extras': {'signature': 'El4KXAERTTIP82L0fQaYLMnvg25m0BZMxh11wd1HJR6qVkFifN45vkyM7N/yP5HE+L9n/VjVD4h+sfU+aOcPP7gBb+1wWJOV9ahzC6+8H9iMRaqNBogmXFaIiI/1Ro6K'}}]

Direct metadata-filtered retrieval:
[metadata={'source': 'learning_dataset', 'topic': 'RAG', 'doc_id'

## 12. Memory Test

The same `thread_id` is reused, so the agent can use the previous conversation state.

This demonstrates the memory concept from the original notebook without needing to search the dataset again for a simple follow-up.


In [22]:
# ============================================================
# 12. FOLLOW-UP / MEMORY TEST
# ============================================================

follow_up = (
    "Now summarize the previous answer in exactly two short bullet points."
)

result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": follow_up}
        ]
    },
    config=config,
)

print(result["messages"][-1].content)


E:\AI-ML\.venv310\lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': '* **Retrieval:** The system searches external documents or databases to find relevant facts for your question.\n* **Generation:** The AI uses those specific facts to write a clear, accurate, and up-to-date answer.', 'extras': {'signature': 'El4KXAERTTIPP4XPzVurkPpX56qPuZRwwJhb0DY/Fv79d7Yxn3wZjLba7+S7huQGR9ceaxzVehn+sJeEwa81a2nRptTgpw8gWqI/TiLEDa6JvXafeCmdRiZo8sokqPBQ'}}]


## 13. Current-Time Test

This uses the actual Python runtime clock rather than a hard-coded timestamp.


In [23]:
# ============================================================
# 13. CURRENT TIME TEST
# ============================================================

time_result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the current date and time right now?"
            }
        ]
    },
    config=config,
)

print(time_result["messages"][-1].content)


E:\AI-ML\.venv310\lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
E:\AI-ML\.venv310\lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'The current date and time is August 25, 2026, at 3:11:11 PM (Nepal Standard Time).', 'extras': {'signature': 'El4KXAERTTIPphLQi2nzcvFeOnMDhY0YKAIs8XfNu+YVg/w5sRch+udWJPaIz+VoJkq0bV4JqJ5G4ZDaUWyk0QuHW68/oum4oqUvQGeRv2TXG4U8/l9YkJ+aDRvMseEE'}}]


## 14. Useful Direct Chroma Tests

These tests are intentionally outside the agent. They make the underlying RAG system easier to understand before you build more advanced agents.


In [24]:
# ============================================================
# 14. DIRECT CHROMA SEARCH TESTS
# ============================================================

print("=== Similarity Search ===")

results = vectorstore.similarity_search(
    "What is a vector database?",
    k=2,
)

for result in results:
    print("\nMetadata:", result.metadata)
    print("Text:", result.page_content)

print("\n=== Finance Filter + Similarity Search ===")

results = vectorstore.similarity_search(
    "financial amount",
    k=3,
    filter={"topic": "Finance"},
)

for result in results:
    print("\nMetadata:", result.metadata)
    print("Text:", result.page_content)


=== Similarity Search ===

Metadata: {'doc_id': 'vec-001', 'source': 'learning_dataset', 'topic': 'Vector Database'}
Text: Vector databases store vector representations of information and support similarity-based retrieval. Chroma can store documents, embeddings, IDs, and metadata for retrieval.

Metadata: {'doc_id': 'vec-001', 'topic': 'Vector Database', 'source': 'learning_dataset'}
Text: Vector databases store vector representations of information and support similarity-based retrieval. Chroma can store documents, embeddings, IDs, and metadata for retrieval.

=== Finance Filter + Similarity Search ===

Metadata: {'source': 'learning_dataset', 'topic': 'Finance', 'doc_id': 'fin-001'}
Text: Acme Learning reported total revenue of 2500000 dollars for the example fiscal year. This number is synthetic training data created for this notebook.

Metadata: {'source': 'learning_dataset', 'topic': 'Finance', 'doc_id': 'fin-001'}
Text: Acme Learning reported total revenue of 2500000 dollars for

## 15. What You Built

### Data layer
- LangChain `Document`
- metadata
- chunking

### Retrieval layer
- Gemini embeddings
- persistent Chroma
- similarity search
- metadata filtering

### Agent layer
- Gemini chat model
- `create_agent`
- tools
- checkpointer/memory
- streaming

### Tools
- dataset RAG search
- calculator
- real-time clock

### Important mental model

`Dataset → Documents → Chunks → Embeddings → Chroma → Retriever → Tool → Agent`

This is the clean foundation for the next stage: more advanced Chroma metadata queries, RAG evaluation with RAGAS, and a larger real-world dataset.
